# Notebook 4: Retrieval, ESRS Context, and Answer Generation

## Purpose
Demonstrate the complete RAG pipeline end-to-end:
1. **Retrieval**: Query chunks across indices
2. **ESRS Context**: Extract relevant disclosure requirements
3. **Generation**: LLM answer generation with proper citations
4. **Validation**: Verify page references and quality

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import json
import pandas as pd

from rag.config import COMPANY_MAP, ESRS_STANDARDS, get_default_pipeline_config
from rag.pipeline import build_index
from rag.retriever import Retriever

print(f'ESRS Standards: {len(ESRS_STANDARDS)}')
print(f'Companies: {len(COMPANY_MAP)}')

ESRS Standards: 12
Companies: 17


In [2]:
SMOKE_TEST = True
config = get_default_pipeline_config()
print(f'Strategy: {config["chunking_strategy"]}')
print(f'Retrieval: {config["retrieval_method"]}')

Strategy: structure_aware
Retrieval: hybrid


In [3]:
print('Building pipeline...')
retriever = None
try:
    vector_idx, bm25_idx, chunks = build_index(config=config, force_rebuild=False)
    retriever = Retriever(
        vector_index=vector_idx,
        bm25_index=bm25_idx,
        chunks=chunks,
        method=config.get('retrieval_method', 'hybrid'),
        top_k=config.get('top_k', 15),
        use_reranker=config.get('use_reranker', False),
    )
    print(f'Ready: {len(chunks)} chunks indexed')
except Exception as e:
    print(f'Warning: {e}')

Building pipeline...
  Loading cached chunks from chunks_structure_aware.json...
    Loaded 658 chunks
  Qdrant init failed (Storage folder /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/artifacts/qdrant_storage is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.), falling back to local store
    Loaded 7315 vectors from local store 'esrs_reports_structure_aware'
  Vector index 'esrs_reports_structure_aware' already has 7315 vectors
  Loading BM25 index from bm25_index_structure_aware.json...
    Loaded BM25 index with 7315 documents


## Select Query Parameters

In [4]:
query_text = 'What are the company\'s greenhouse gas emissions and reduction targets?'
company_name = 'Shell'
esrs_standard = 'ESRS E1'

print(f'Query: {query_text}')
print(f'Company: {company_name}')
print(f'Standard: {esrs_standard}')

Query: What are the company's greenhouse gas emissions and reduction targets?
Company: Shell
Standard: ESRS E1


## Step 1: Retrieval

In [5]:
print(f'Retrieving passages...\n')

if retriever is not None:
    try:
        results = retriever.retrieve(query_text, top_k=5)
        print(f'Retrieved {len(results)} passages:\n')
        
        for i, result in enumerate(results, 1):
            chunk = result.get('chunk')
            score = result.get('score', 0)
            page_info = f'p. {chunk.page_number}' if chunk.page_number >= 0 else 'unknown'
            preview = chunk.text[:100].replace('\\n', ' ') + '...'
            print(f'{i}. [{page_info}] Score: {score:.3f}')
            print(f'   {preview}')
    except Exception as e:
        print(f'Retrieval error: {e}')
else:
    print('Retriever not available')

Retrieving passages...

Retriever not available


## Step 2: ESRS Context

In [6]:
from rag.esrs_context import get_esrs_requirements_text

print(f'Extracting ESRS requirements for {esrs_standard}...\n')

try:
    esrs_text = get_esrs_requirements_text(esrs_standard)
    preview = esrs_text[:200].replace('\\n', ' ') if esrs_text else 'None'
    print(f'Requirements: {preview}...')
except Exception as e:
    print(f'Note: ESRS extraction not available: {e}')

Extracting ESRS requirements for ESRS E1...

Note: ESRS extraction not available: get_esrs_requirements_text() missing 1 required positional argument: 'esrs_standard'


## Step 3: Answer Generation

In [7]:
from rag.generator import generate_answer

print(f'Generating answer...\n')

if retriever is None:
    print('Cannot generate: Retriever not initialized')
else:
    try:
        passages = retriever.retrieve(query_text, top_k=10)
        context_passages = [f'Source {i+1}: {p["chunk"].text}' for i, p in enumerate(passages)]
        
        answer = generate_answer(
            query=query_text,
            context_passages=context_passages,
            esrs_standard=esrs_standard,
            company=company_name,
        )
        
        if answer:
            print('Answer generated successfully')
            if isinstance(answer, dict):
                print(json.dumps(answer, indent=2)[:500])
        else:
            print('No answer generated')
    except Exception as e:
        print(f'Note: Generation not available: {e}')

Generating answer...

Cannot generate: Retriever not initialized


## Citation Resolution

In [8]:
print('Citation resolution example:')
print()

if retriever is not None:
    try:
        results = retriever.retrieve(query_text, top_k=3)
        print(f'Top {len(results)} results with page references:')
        print()
        
        for i, result in enumerate(results, 1):
            chunk = result.get('chunk')
            page_nums = chunk.page_span if chunk.page_span else [chunk.page_number]
            page_str = f'pp. {page_nums[0]}-{page_nums[-1]}' if len(page_nums) > 1 else f'p. {page_nums[0]}'
            print(f'[Citation {i}] {page_str}')
            print(f'Company: {chunk.company}')
            print(f'Headers: {" > ".join(chunk.header_hierarchy[:2])}')
            print()
    except Exception as e:
        print(f'Error: {e}')

Citation resolution example:



## Multi-Company Comparison (Optional)

In [9]:
if not SMOKE_TEST and retriever is not None:
    print('Comparing 2 companies...\n')
    
    companies_to_compare = list(COMPANY_MAP.keys())[:2]
    
    for company in companies_to_compare:
        try:
            results = retriever.retrieve(query_text, top_k=3)
            print(f'{COMPANY_MAP[company]}: {len(results)} relevant passages')
        except:
            pass
else:
    print('Comparison skipped in smoke test mode')

Comparison skipped in smoke test mode
